## 1. Подготовка среды


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
local_deps = ROOT.parents[1] / "work/pr2/deps"
if local_deps.exists():
    sys.path.insert(0, str(local_deps))
sys.path.insert(0, str(ROOT))
from scripts import eda, diagrams
from importlib import reload
reload(eda)
reload(diagrams)
print("Среда готова.")
print("Корень проекта:", ROOT)


Практическая работа № 2. Среда готова.
Корень проекта: c:\Users\user\Documents\Codex\2026-10-25\gh\outputs\practical2_project


## 2. Разбиение по времени


In [10]:
import contextlib, io
from scripts import split_data
sys.argv = ["split_data.py", "--root", str(ROOT)]
with contextlib.redirect_stdout(io.StringIO()):
    split_data.main()
manifest = json.loads((ROOT / "split_manifest.json").read_text())
for part in manifest["partitions"]:
    print(part["file"], part["rows"], part["searchDate_min"], part["searchDate_max"])
print("Проверки:", manifest["validation"])


data/initial/initial.csv 1798 2022-04-16 2022-07-31
data/incoming/batch_01.csv 397 2022-08-01 2022-08-21
data/incoming/batch_02.csv 395 2022-08-22 2022-09-12
data/incoming/batch_03.csv 410 2022-09-13 2022-10-05
Проверки: {'all_rows_preserved': True, 'values_preserved': True, 'dates_do_not_overlap': True, 'raw_unchanged': True}


## 3. Структура начальной части


In [11]:
df = eda.dataset_summary()
print("\nОбъём начальной части:", df.shape[0], "записей")
print("Модель в этой работе не обучается.")


{
  "rows": 1798,
  "source_columns": 20,
  "search_min": "2022-04-16",
  "search_max": "2022-07-31",
  "flight_min": "2022-04-18",
  "flight_max": "2022-09-25",
  "routes": 233,
  "airline_itineraries": 45,
  "cabin_counts": {
    "coach": 1795,
    "mixed": 2,
    "premium coach": 1
  }
}

Объём начальной части: 1798 записей
Модель в этой работе не обучается.


## 4. Проверка качества и целевой переменной


In [12]:
quality = eda.quality_report()
eda.price_plots()


Начальная часть: 1798 записей
Пропуски расстояния: 141 (7.84%)
Полные дубликаты: 0
Дубликаты ключа наблюдения: 0
Нарушений проверок допустимости: 0
Выбросы цены по IQR: 15 ; верхняя граница 906.93 USD
count    1798.00
mean      362.64
std       194.43
min        46.58
25%       209.71
50%       328.60
75%       488.60
95%       704.75
99%       892.62
max      1771.59
График сохранён: results/figures/price.png


## 5. Влияние срока до вылета, расстояния, пересадок и класса


In [13]:
eda.relationship_plots()
print("Графики зависимостей и групповых медиан сохранены.")


Окна бронирования:
                 count  median    mean
booking_window                       
0–7               259   342.6  373.66
8–14              235   320.1  351.89
15–30             485   351.6  383.07
31+               819   316.7  350.15

Пересадки:
              count  median
stops_count               
0              520  218.60
1             1112  354.70
2              161  567.11
3                5  660.05

Классы:
                count  median
cabin                       
coach           1795   328.6
mixed              2  1137.1
premium coach      1  1128.6
Графики зависимостей и групповых медиан сохранены.


## 6. Взаимосвязи между признаками


In [14]:
correlations = eda.correlation_report()


Корреляции с ценой:
 totalFare              1.000
days_before_flight    -0.052
totalTravelDistance    0.627
duration_minutes       0.544
stops_count            0.433
seatsRemaining         0.006
flight_month          -0.098
flight_day_of_week     0.109

Расстояние и длительность: 0.75


## 7. Проверка предметных гипотез


In [15]:
hypotheses = eda.hypothesis_report()


H1 Цена при сроке до вылета не более 7 дней выше, чем при сроке от 30 дней
  n=259/841; медианы=342.60/318.60 USD; p Holm=0.0658
  Поддерживается в EDA: False
H2 С увеличением расстояния цена повышается
  n=1657; rho=0.627; p Holm=4.36e-181
  Поддерживается в EDA: True
H3 Предложения с пересадками дороже прямых в общей выборке
  n=1278/520; медианы=377.40/218.60 USD; p Holm=3.68e-52
  Поддерживается в EDA: True
Результаты разведочные: состав маршрутов и классов, повторные наблюдения не контролируются.


## 8. Проектирование архитектуры и пайплайна


In [16]:
diagrams.main()
eda.artifact_inventory()
print("Разбиение, разведочный анализ и диаграммы завершены.")


Созданы 5 диаграмм C4/UML в results/figures (PNG и SVG).
Сохранено результатов: 27
Разбиение, разведочный анализ и диаграммы завершены.
